In [1]:
# ========================
# PARTE 0: INSTALAÇÕES (Google Colab)
# ========================
!apt-get update > /dev/null
!apt install chromium-chromedriver -y > /dev/null || true
!cp /usr/lib/chromium-browser/chromedriver /usr/bin || true
!pip install selenium > /dev/null
!pip install pandas openpyxl > /dev/null

# ========================
# PARTE 1: IMPORTS E CONFIGURAÇÕES
# ========================
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
import pandas as pd
import os, time, zipfile, requests
from datetime import datetime

# Selenium Headless
options = Options()
options.binary_location = "/usr/bin/chromium-browser"
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

driver = webdriver.Chrome(options=options)
wait = WebDriverWait(driver, 15)

# ========================
# PARTE 2: COLETAR LINKS DE PRODUTOS
# ========================
url = "https://www.leroymerlin.com.br/search?term=Revestimentos%20de%20Parede"
driver.get(url)

todos_links = set()
pagina = 1
LIMITE_PRODUTOS = 10

while len(todos_links) < LIMITE_PRODUTOS:
    print(f"🔄 Página {pagina}...")
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "a[href*='/revestimento']")))
    except TimeoutException:
        print("⚠️ Timeout esperando os produtos.")
        break

    produtos = driver.find_elements(By.CSS_SELECTOR, "a[href*='/revestimento']")
    for p in produtos:
        link = p.get_attribute("href")
        if link:
            todos_links.add(link)
            if len(todos_links) >= LIMITE_PRODUTOS:
                break

    try:
        botao_proximo = driver.find_element(By.CSS_SELECTOR, "button[aria-label='Próxima página']")
        if not botao_proximo.is_enabled():
            break
        driver.execute_script("arguments[0].click();", botao_proximo)
        pagina += 1
        time.sleep(2)
    except NoSuchElementException:
        break

print(f"\n✅ Total de links coletados: {len(todos_links)}\n")

# ========================
# PARTE 3: EXTRAÇÃO DE DADOS
# ========================
def extrair_dados_produto(url):
    driver.get(url)
    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "h1")))
    except TimeoutException:
        print(f"⚠️ Timeout em: {url}")
        return {'URL': url, 'Erro': 'Timeout'}

    dados = {'URL': url}

    try:
        dados['Nome'] = driver.find_element(By.CSS_SELECTOR, "h1").text
    except:
        dados['Nome'] = None

    try:
        dados['Código'] = url.split("_")[-1]
    except:
        dados['Código'] = None

    try:
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, '[data-cy="product-price-integer"]')))
        inteiro = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-integer"]').text
        decimal = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-decimal"]').text
        dados['Preço Promocional'] = f"{inteiro},{decimal}"
    except:
        dados['Preço Promocional'] = "N/A"

    try:
        dados['Preço Original'] = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-from-price"]').text
    except:
        dados['Preço Original'] = "N/A"

    try:
        embalagem = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-packaging"] strong').text
        unidade = driver.find_element(By.CSS_SELECTOR, '[data-cy="product-price-packaging"] span:nth-child(2)').text
        dados['Preço por m²'] = f"{embalagem} | {unidade}"
    except:
        dados['Preço por m²'] = "N/A"

    try:
        dados['Descrição'] = driver.find_element(By.CSS_SELECTOR, 'meta[name="description"]').get_attribute("content")
    except:
        dados['Descrição'] = None

    try:
        dados['Imagem'] = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:image"]').get_attribute("content")
    except:
        dados['Imagem'] = "N/A"

    try:
        especificacoes = {}
        linhas = driver.find_elements(By.CSS_SELECTOR, 'table tr')
        for linha in linhas:
            try:
                chave = linha.find_element(By.CSS_SELECTOR, 'th').text.strip()
                valor = linha.find_element(By.CSS_SELECTOR, 'td').text.strip()
                especificacoes[chave] = valor
            except:
                continue
        dados['Especificações'] = "\n".join([f"{k}: {v}" for k, v in especificacoes.items()])
    except:
        dados['Especificações'] = None

    dados['Loja'] = "Leroy Merlin"
    return dados

# ========================
# PARTE 4: COLETAR TODOS OS PRODUTOS
# ========================
todos_produtos = []
for idx, link in enumerate(sorted(todos_links), 1):
    print(f"🔍 Coletando {idx}/{len(todos_links)}: {link}")
    try:
        dados = extrair_dados_produto(link)
        todos_produtos.append(dados)
    except Exception as e:
        print(f"⚠️ Erro ao coletar {link}: {e}")

driver.quit()

# ========================
# PARTE 5: SALVAR CSV / EXCEL
# ========================
data_hoje = datetime.now().strftime("%Y-%m-%d")
pasta_base = "saida_leroy"
os.makedirs(pasta_base, exist_ok=True)

csv_path = os.path.join(pasta_base, f"produtos_pisos_revestimentos_leroy_{data_hoje}.csv")
xlsx_path = os.path.join(pasta_base, f"produtos_pisos_revestimentos_leroy_{data_hoje}.xlsx")

df = pd.DataFrame(todos_produtos)
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
df.to_excel(xlsx_path, index=False)

# ========================
# PARTE 6: BAIXAR IMAGENS COM USER-AGENT
# ========================
images_dir = os.path.join(pasta_base, f"imagens_leroy_{data_hoje}")
os.makedirs(images_dir, exist_ok=True)

for _, row in df.iterrows():
    codigo = row["Código"]
    img_url = row["Imagem"]
    if img_url != "N/A":
        tentativas, sucesso = 0, False
        while tentativas < 3 and not sucesso:
            try:
                headers = {
                    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
                }
                resp = requests.get(img_url, headers=headers, stream=True, timeout=10)
                if resp.status_code == 200:
                    img_path = os.path.join(images_dir, f"{codigo}.png")
                    with open(img_path, "wb") as f:
                        for chunk in resp.iter_content(1024):
                            f.write(chunk)
                    sucesso = True
                else:
                    print(f"❌ Erro imagem {codigo}: Status {resp.status_code}")
            except Exception as e:
                print(f"❌ Erro imagem {codigo}: {e}")
            finally:
                if not sucesso:
                    tentativas += 1
                    if tentativas < 3:
                        print(f"🔁 Tentando novamente {codigo} (tentativa {tentativas + 1})...")
                        time.sleep(5)

# ========================
# PARTE 7: COMPACTAR IMAGENS
# ========================
zip_filename = os.path.join(pasta_base, f"imagens_leroy_{data_hoje}.zip")
with zipfile.ZipFile(zip_filename, "w", zipfile.ZIP_DEFLATED) as zipf:
    for filename in os.listdir(images_dir):
        zipf.write(os.path.join(images_dir, filename), filename)

# ========================
# PARTE 8: MENSAGENS FINAIS
# ========================
print(f"\n📄 CSV salvo: {csv_path}")
print(f"📊 Excel salvo: {xlsx_path}")
print(f"📦 Imagens compactadas: {zip_filename}")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


cp: '/usr/lib/chromium-browser/chromedriver' and '/usr/bin/chromedriver' are the same file
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which 